In [ ]:
!pip install -q torch transformers accelerate datasets wandb peft
!mkdir -p /content/peft_output
import os
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader
from transformers import default_data_collator

# Configuration
MODEL_CHECKPOINT = "HuggingFaceTB/SmolLM2-1.7B"
OUTPUT_DIR = "/content/peft_output"
LEARNING_RATE = 1e-4
BATCH_SIZE = 4
NUM_EPOCHS = 5
MAX_SEQUENCE_LENGTH = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EuclideanBoundarySpace:
    def __init__(self, dimension=3, boundary_min=0, boundary_max=10):
        """
        Create a Euclidean boundary space with specified dimensions and limits.

        Args:
            dimension (int): Number of spatial dimensions
            boundary_min (float): Minimum boundary value
            boundary_max (float): Maximum boundary value
        """
        self.dimension = dimension
        self.boundary_min = boundary_min
        self.boundary_max = boundary_max

    def generate_point(self):
        """
        Generate a random point within the Euclidean boundary space.

        Returns:
            torch.Tensor: A point within the defined boundaries
        """
        return torch.tensor(
            np.random.uniform(
                low=self.boundary_min,
                high=self.boundary_max,
                size=self.dimension
            ),
            dtype=torch.float32
        )

    def is_inside(self, point):
        """
        Check if a point is inside the boundary space.

        Args:
            point (torch.Tensor): Point to check

        Returns:
            bool: True if point is inside, False otherwise
        """
        return torch.all(
            (point >= self.boundary_min) &
            (point <= self.boundary_max)
        )

# Provided training examples
spatial_prompts = [
    "(o ) → Is ONE element INSIDE?",
    "|x x x| → Are ALL elements INSIDE?",
    "o|x| → Is 'x' INSIDE or OUTSIDE?",
    "o|x| → Is 'o' INSIDE or OUTSIDE?",
    "|o|x → Is 'o' INSIDE or OUTSIDE?",
    "| x| → Is ONE element INSIDE?",
    "|o | → Is ONE element INSIDE?",
    "|oo| → Are TWO elements INSIDE?",
    "|o| → Is ONE element INSIDE?",
    "|o| → Are TWO elements INSIDE?",
    "$|$#| → Are SOME elements INSIDE?",
    "x|ox| → Are SOME elements INSIDE?",
    "|oo |o → Are SOME elements INSIDE?",
    "x|ox| → Are ALL elements INSIDE?",
    "|ox|o → Are ALL elements INSIDE?",
    "|xox| → Are ALL elements INSIDE?",
    "|xx x| → Are ALL elements INSIDE?",
    "|oo|o → Are ALL elements INSIDE?",
    "x|| → Is at least one element INSIDE?",
    "|||x → Is at least one element INSIDE?",
    "|o|| → Is at least one element INSIDE?",
    "||o| → Is at least one element INSIDE?",
    "|||o → Is 'o' INSIDE or OUTSIDE?",
    "o||| → Is 'o' INSIDE or OUTSIDE?",
    "| o| → Is 'o' INSIDE or OUTSIDE?",
    "|o | → Is 'o' INSIDE or OUTSIDE?",
    "a|| → Is 'a' INSIDE or OUTSIDE?",
    "||a → Is 'a' INSIDE or OUTSIDE?",
    "|($)| → Is '$' INSIDE or OUTSIDE?",
    "o||| → Is the object INSIDE or OUTSIDE?",
    "||||x → Is the object INSIDE or OUTSIDE?",
    "x|||| → Is the object INSIDE or OUTSIDE?",
    "| o | → Is the object INSIDE or OUTSIDE?",
    "| x | → Is the object INSIDE or OUTSIDE?",
    "||o|| → Is the object INSIDE or OUTSIDE?",
    "||x|| → Is the object INSIDE or OUTSIDE?",
    "|o o| vs ||o||o → Are all elements INSIDE?",
    "|o o| vs ||o o|| → Are all elements INSIDE?",
    "|o→o| → What happens when a boundary becomes partially open?",
    "|x|| → Is at least one element INSIDE?",
    "|$$#| → Are ALL elements INSIDE?",
    "o|o| → What defines the boundary here?",
    "|x x| → Are BOTH elements INSIDE?",
    "||x → Are ALL elements INSIDE?",
    "|%$| → Are ALL elements INSIDE?",
    "||x → Is 'x' INSIDE or OUTSIDE?",
    "||||o → Is 'o' INSIDE or OUTSIDE?",
    "||o|| → Is 'o' INSIDE or OUTSIDE?",
    "x|| → Is 'x' INSIDE or OUTSIDE?",
    "|x| → Is 'x' INSIDE or OUTSIDE?",
    "|o| → Is 'o' INSIDE or OUTSIDE?",
    "o|| → Is 'o' INSIDE or OUTSIDE?",
    "||o → Is 'o' INSIDE or OUTSIDE?",
    "|oo| → How many elements are INSIDE?",
    "|o o| → Are BOTH elements INSIDE?",
    "| o| → Is 'o' INSIDE or OUTSIDE?",
    "|oo| → Are ALL elements INSIDE?",
    "||o|| → Is 'o' INSIDE or OUTSIDE?",
    "|o o o| → How many elements are INSIDE?",
    "|oo|oo| → How many elements are INSIDE?",
    "oo| → Is 'o' INSIDE or OUTSIDE?",
    "|oo| → Are ALL elements OUTSIDE?",
    "|o|| → Is at least one element INSIDE?",
    "| o o | → How many elements are INSIDE?",
    "|o o o| → Are ALL elements INSIDE?",
    "| | → How many elements are INSIDE?",
    "|ooo| → Are ALL elements INSIDE?",
    "|| → Is anything INSIDE?",
    "|oo o| → Are ALL elements INSIDE?",
    "|o| → How many elements are INSIDE?",
    "|o| → ||| → What occurs when a boundary collapses?",
    "||o|o|| → Determine relative interior/exterior relationships",
    "|#@o$| → Are ALL elements INSIDE?",
    "(o o) → Are BOTH elements INSIDE?",
    "░o░o░ → Are BOTH elements INSIDE?",
    "|(o)| → Is 'o' INSIDE the inner boundary?",
    "|o[x]| → Is 'x' INSIDE the inner boundary but OUTSIDE the outer boundary?",
    "|oxo| → Is 'x' to the LEFT or RIGHT of 'o'?",
    "|oo|  |xx| → How many elements are in BOTH boundaries?",
    "|o x| → If 'o' moves to the RIGHT, is it still INSIDE?",
    "|o o x o x o| → How many 'x' are INSIDE?",
    "| o  | → Is 'o' INSIDE? (Consider the spacing)",
    "||oo|o| → Describe the spatial arrangement of the elements.",
    "(o) → Is 'o' INSIDE or OUTSIDE?",
    "( o ) → Is 'o' INSIDE or OUTSIDE?",
    "(x) → Is 'x' INSIDE or OUTSIDE?",
    "o() → Is 'o' INSIDE or OUTSIDE?",
    "()o → Is 'o' INSIDE or OUTSIDE?",
    "(a) → Is 'a' INSIDE or OUTSIDE?",
    "()a → Is 'a' INSIDE or OUTSIDE?",
    "|(o)| → Is 'o' INSIDE or OUTSIDE?",
    "|(x)| → Is 'x' INSIDE or OUTSIDE?",
    "|xxx| → Are ALL elements INSIDE?",
    "|o o| → Are ALL elements INSIDE?",
    "|oox| → If 'x' moves LEFT, what is the new arrangement?",
    "|o o| → If the elements SWAP positions, what happens?",
    "|oo| → If the boundary grows to the RIGHT, what is the new arrangement?",
    "|^o| → If 'o' rotates 90 degrees CLOCKWISE, describe the new arrangement.",
    "|oxx| → If the LEFTMOST element is 'o', are ALL elements INSIDE?",
    "|#o$| → If '#' is INSIDE, is '$' also INSIDE?",
    "|oxo| → Is 'x' INSIDE?",
    "ooo| → Are the 'o's INSIDE?",
]

correct_responses = [
    "YES",
    "YES",
    "INSIDE",
    "OUTSIDE",
    "INSIDE",
    "YES",
    "YES",
    "YES",
    "YES",
    "NO",
    "YES",
    "YES",
    "YES",
    "NO",
    "NO",
    "YES",
    "YES",
    "NO",
    "NO",
    "NO",
    "YES",
    "YES",
    "OUTSIDE",
    "OUTSIDE",
    "INSIDE",
    "INSIDE",
    "OUTSIDE",
    "OUTSIDE",
    "INSIDE",
    "OUTSIDE",
    "OUTSIDE",
    "OUTSIDE",
    "INSIDE",
    "INSIDE",
    "INSIDE", "INSIDE", "NO", "YES",
    "The object moves from inside to outside.", "YES", "YES",
    "| defines the boundary.", "YES", "NO", "YES", "OUTSIDE",
    "OUTSIDE", "INSIDE", "OUTSIDE", "INSIDE", "INSIDE", "OUTSIDE",
    "OUTSIDE", "2", "YES", "INSIDE", "YES", "OUTSIDE", "3", "2",
    "OUTSIDE", "NO", "YES", "2", "YES", "0", "YES", "NO", "NO",
    "1", "The object moves from inside to outside.",
    "There are two objects inside.", "YES", "YES", "YES", "YES",
    "YES", "LEFT", "0", "YES", "2", "YES",
    "All elements are INSIDE.",
    "INSIDE",
    "INSIDE",
    "INSIDE",
    "OUTSIDE",
    "OUTSIDE",
    "INSIDE",
    "OUTSIDE",
    "INSIDE",
    "INSIDE",
    "YES",
    "YES",
    "|xoo|",
    "|o o| - No change",
    "|oo  |",
    "|>o|",
    "YES",
    "YES",
    "YES",
    "NO",
]

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_CHECKPOINT).to(DEVICE)

# Configure LoRA for PEFT
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
)
peft_model = get_peft_model(base_model, lora_config)

# Create Dataset
dataset = Dataset.from_dict({
    "prompts": spatial_prompts,
    "responses": correct_responses
})

def preprocess_function(examples):
    inputs = examples["prompts"]
    targets = examples["responses"]
    model_inputs = tokenizer(
        inputs, truncation=True, padding="max_length", max_length=MAX_SEQUENCE_LENGTH
    )
    labels = tokenizer(
        targets, truncation=True, padding="max_length", max_length=MAX_SEQUENCE_LENGTH
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)
data_collator = default_data_collator
dataloader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, collate_fn=data_collator)

# Optimizer
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=LEARNING_RATE)

# Training loop
peft_model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = peft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {avg_loss}")

# Save model
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Inference
peft_model.eval()
with torch.no_grad():
    # Use a subset of the original prompts for testing
    test_prompts = spatial_prompts[:5]

    for test_prompt in test_prompts:
        inputs = tokenizer(test_prompt, return_tensors="pt",
                           padding="max_length",
                           max_length=MAX_SEQUENCE_LENGTH).to(DEVICE)

        outputs = peft_model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=200,
            num_beams=5,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
        )
        print(f"Prompt: {test_prompt}")
        print("Generated Response:", tokenizer.decode(outputs[0], skip_special_tokens=True))
        print("---")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/102 [00:00<?, ? examples/s]

Epoch 1/5, Loss: 0.9550863962907058
Epoch 2/5, Loss: 0.38995242290771925
Epoch 3/5, Loss: 0.16650554079275864
Epoch 4/5, Loss: 0.09808857481066997
Epoch 5/5, Loss: 0.06775750477726643
Prompt: (o ) → Is ONE element INSIDE?
Generated Response: (o ) → Is ONE element INSIDE?
---
Prompt: |x x x| → Are ALL elements INSIDE?
Generated Response: |x x x| → Are ALL elements INSIDE?
---
Prompt: o|x| → Is 'x' INSIDE or OUTSIDE?
Generated Response: o|x| → Is 'x' INSIDE or OUTSIDE?IDE
---
Prompt: o|x| → Is 'o' INSIDE or OUTSIDE?
Generated Response: o|x| → Is 'o' INSIDE or OUTSIDE?IDE
---
Prompt: |o|x → Is 'o' INSIDE or OUTSIDE?
Generated Response: |o|x → Is 'o' INSIDE or OUTSIDE?NOO
---


In [ ]:
# Improved Inference with Logic
peft_model.eval()
test_prompt = (
   "|h| → Is 'o' INSIDE or OUTSIDE?"
)
inputs = tokenizer(test_prompt, return_tensors="pt", padding="max_length", max_length=MAX_SEQUENCE_LENGTH).to(DEVICE)
outputs = peft_model.generate(
    inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_length=200,
    num_beams=5,
    early_stopping=True,
    pad_token_id=tokenizer.pad_token_id,
)
print("Generated Response:", tokenizer.decode(outputs[0], skip_special_tokens=True))

Generated Response: |h| → Is 'o' INSIDE or OUTSIDE?IDE


**This is the Reinforcement Learning method. ChatGPT swears up and down this should be better. I cannot train the model this way and not get model collapse. I get model collapse 100% of the time. I do not know why.**

In [ ]:
!pip install -q torch transformers accelerate datasets wandb peft
!mkdir -p /content/peft_output
import os
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader
from transformers import default_data_collator
from difflib import SequenceMatcher

# Configuration
MODEL_CHECKPOINT = "HuggingFaceTB/SmolLM2-1.7B"
OUTPUT_DIR = "/content/peft_output"
LEARNING_RATE = 1e-5
BATCH_SIZE = 1
NUM_EPOCHS = 5
MAX_SEQUENCE_LENGTH = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_CHECKPOINT).to(DEVICE)

# Configure LoRA for PEFT
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
)
peft_model = get_peft_model(base_model, lora_config)

# Create Dataset
dataset = Dataset.from_dict({"prompts": spatial_prompts, "responses": correct_responses})

def preprocess_function(examples):
    inputs = examples["prompts"]
    targets = examples["responses"]
    model_inputs = tokenizer(
        inputs, truncation=True, padding="max_length", max_length=MAX_SEQUENCE_LENGTH
    )
    labels = tokenizer(
        targets, truncation=True, padding="max_length", max_length=MAX_SEQUENCE_LENGTH
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)
data_collator = default_data_collator
dataloader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, collate_fn=data_collator)

# Optimizer
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=LEARNING_RATE)

# Reward Function
def calculate_reward(generated, target):
    """
    Calculate reward based on spatial reasoning correctness.
    Rewards partial correctness and penalizes repetition.
    """
    generated = generated.strip()
    target = target.strip()

    # Structural similarity (e.g., Levenshtein distance)
    similarity = SequenceMatcher(None, generated, target).ratio()

    # Penalize repetition
    repetition_penalty = -0.1 if len(set(generated.split())) < len(generated.split()) else 0.0

    # Reward scaling: Perfect match = 1.0, Partial matches scaled
    reward = similarity + repetition_penalty
    return reward

# Training Loop
peft_model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # Forward pass
        outputs = peft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        logits = outputs.logits

        # Generate response
        probs = torch.softmax(logits, dim=-1)
        m = torch.distributions.Categorical(probs)
        actions = m.sample()
        generated_response = tokenizer.decode(actions[0], skip_special_tokens=True)

        # Compute reward
        correct_response = tokenizer.decode(labels[0], skip_special_tokens=True)
        reward = calculate_reward(generated_response, correct_response)

        # Policy gradient loss
        log_probs = m.log_prob(actions)
        loss = -log_probs.mean() * reward

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {avg_loss}")

# Save Model
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Inference
peft_model.eval()
with torch.no_grad():
    test_prompts = spatial_prompts[:5]  # Subset for testing
    for test_prompt in test_prompts:
        inputs = tokenizer(test_prompt, return_tensors="pt", padding="max_length", max_length=MAX_SEQUENCE_LENGTH).to(DEVICE)
        outputs = peft_model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=50,
            num_beams=3,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
        )
        print(f"Prompt: {test_prompt}")
        print("Generated Response:", tokenizer.decode(outputs[0], skip_special_tokens=True))
        print("---")